# 🔬 DeepTrace v2 — EfficientNet-B4 Fine-Grained Training Pipeline

This notebook trains an **EfficientNet-B4** model for deepfake detection using the
**[ScaleDF](https://huggingface.co/datasets/WenhaoWang/ScaleDF)** dataset with
**134 fine-grained classes** (46 real sources + 88 fake generation methods).

### Why Fine-Grained (134 classes) instead of Binary (Real/Fake)?
- **Source Attribution**: The model can identify *which* generation method was used (e.g., StableDiffusion, AniPortrait)
- **Better Detection**: Learning method-specific artifacts prevents mode collapse
- **Richer Forensics**: The backend can report confidence per generation method
- **Binary compatibility**: Fake probability = sum of all 88 fake class probabilities

The data **streams directly from HuggingFace** via `webdataset` — no manual download needed.

### ⚡ Prerequisites
- Set your Kaggle/Colab Runtime to **T4 GPU** or **A100 GPU**
- Set your HuggingFace token as a Kaggle Secret named `HF_TOKEN`


In [ ]:
# ── Module 1: Environment Setup ──
# albumentations pinned to 2.0.8 — last MIT-licensed release before AGPL fork.
# opencv-python-headless avoids libGL errors on headless runtimes.
# webdataset streams ScaleDF's tar shards without downloading the full 2TB+ dataset.
!pip install -q transformers datasets accelerate torchvision evaluate scikit-learn \
    "albumentations==2.0.8" opencv-python-headless webdataset huggingface_hub


## 1. Environment, Reproducibility & Authentication

Seeds are fixed for reproducible training. The HuggingFace token is loaded from
Kaggle Secrets (never hardcoded in the notebook).


In [ ]:
# ── Module 2: GPU Detection, Reproducibility & Authentication ──
import os
import io
import random
import numpy as np
import torch
from PIL import Image

# ── Reproducibility ──
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device detection ──
if torch.cuda.is_available():
    device = torch.device("cuda")
    torch.cuda.manual_seed_all(SEED)
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB VRAM)")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    print("WARNING: No GPU detected. Training will be extremely slow.")

# ── HuggingFace Authentication ──
# Try Kaggle Secrets first, then getpass. NEVER hardcode tokens.
try:
    from kaggle_secrets import UserSecretsClient
    _hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF token loaded from Kaggle Secrets.")
except Exception:
    try:
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab Secrets.")
    except Exception:
        import getpass
        _hf_token = getpass.getpass("Enter your HuggingFace token: ")

assert _hf_token, "HF_TOKEN is required to stream ScaleDF."
os.environ["HF_TOKEN"] = _hf_token

# Explicitly authenticate with HF Hub (os.environ alone is not enough)
from huggingface_hub import login
login(token=_hf_token, add_to_git_credential=False)
print("HuggingFace token successfully loaded and authenticated!")


## 2. Load EfficientNet-B4 Processor

The processor carries the exact normalization statistics (`image_mean` / `image_std`) and
input resolution baked into the `google/efficientnet-b4` checkpoint.

> ⚠️ `google/efficientnet-b4`'s `image_std` is `[0.4785, 0.4733, 0.4743]`, **not** the
> generic ImageNet `[0.229, 0.224, 0.225]`. Training with mismatched normalization silently
> degrades every prediction. 


In [ ]:
# ── Module 3: Image Processor & Normalization ──
from transformers import AutoImageProcessor

MODEL_ID = "google/efficientnet-b4"
processor = AutoImageProcessor.from_pretrained(MODEL_ID)

IMAGE_SIZE = processor.size["height"]  # 380 for B4
NORM_MEAN  = processor.image_mean
NORM_STD   = processor.image_std

print(f"Resolution:     {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Normalize mean: {NORM_MEAN}")
print(f"Normalize std:  {NORM_STD}")


## 3. Fine-Grained 134-Class Label Discovery

ScaleDF organizes its data into WebDataset `.tar` shards:
- **Real** shards have filenames starting with `000000` (e.g., `000000AFAD.tar`, `000000FFHQ.tar`)
- **Fake** shards use generation method names (e.g., `StableDiffusion_faces.tar`, `AMatrix_faces.tar`)

We discover all 134 classes dynamically, build label mappings, and separate
real/fake class IDs for computing binary metrics at eval time.


In [ ]:
# ── Module 4: Fine-Grained 134-Class Label Discovery ──
from huggingface_hub import HfApi

api = HfApi()
REPO_ID = "WenhaoWang/ScaleDF"
BASE_URL = "https://huggingface.co/datasets/WenhaoWang/ScaleDF/resolve/main/"
_url_suffix = f"?token={_hf_token}" if _hf_token else ""

# ── Discover training shards ──
print("Discovering training shards...")
train_tree = api.list_repo_tree(REPO_ID, path_in_repo="ScaleDF/train", repo_type="dataset")
train_tars = sorted([item.path for item in train_tree if item.path.endswith(".tar")])

# ── Extract class names from shard filenames ──
class_names = [os.path.basename(t).replace(".tar", "") for t in train_tars]
NUM_CLASSES = len(class_names)
assert NUM_CLASSES > 0, "No training shards found! Check your HF token and network."

# ── Build label mappings ──
label2id = {name: idx for idx, name in enumerate(class_names)}
id2label = {idx: name for idx, name in enumerate(class_names)}

# ── Build binary grouping (for inference-time Real/Fake verdict) ──
real_class_ids = sorted([idx for idx, name in enumerate(class_names) if name.startswith("000000")])
fake_class_ids = sorted([idx for idx, name in enumerate(class_names) if not name.startswith("000000")])
assert len(real_class_ids) + len(fake_class_ids) == NUM_CLASSES

# ── Build training URLs and URL-to-label mapping ──
train_urls = [BASE_URL + t + _url_suffix for t in train_tars]
url_to_label = {}
for t in train_tars:
    shard_name = os.path.basename(t).replace(".tar", "")
    url_to_label[os.path.basename(t)] = label2id[shard_name]

# ── Print summary ──
print(f"\nTotal classes:      {NUM_CLASSES}")
print(f"Real classes:       {len(real_class_ids)}")
print(f"Fake classes:       {len(fake_class_ids)}")
print(f"Training shards:    {len(train_tars)}")
print(f"\nSample real classes: {[id2label[i] for i in real_class_ids[:5]]}")
print(f"Sample fake classes: {[id2label[i] for i in fake_class_ids[:5]]}")


## 4. Forensic Augmentation Pipeline

To survive WhatsApp/Instagram compression, we simulate social media degradation
during training. All transforms use the **Albumentations 2.x** API.


In [ ]:
# ── Module 5: Data Augmentation Pipeline ──
import albumentations as A
from albumentations.pytorch import ToTensorV2

train_augmentations = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.Affine(
        translate_percent=(-0.05, 0.05),
        scale=(0.95, 1.05),
        rotate=(-15, 15),
        p=0.5,
    ),
    A.ColorJitter(
        brightness=(0.8, 1.2), contrast=(0.8, 1.2),
        saturation=(0.8, 1.2), hue=(-0.1, 0.1), p=0.5,
    ),
    # Social media compression simulation
    A.ImageCompression(quality_range=(30, 90), p=0.6),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(std_range=(0.03, 0.12), p=0.3),
    # Partial occlusion (hands, hair, masks)
    A.CoarseDropout(
        num_holes_range=(1, 3),
        hole_height_range=(0.05, 0.15),
        hole_width_range=(0.05, 0.15), p=0.15,
    ),
    A.RandomResizedCrop(size=(IMAGE_SIZE, IMAGE_SIZE), scale=(0.8, 1.0)),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

val_augmentations = A.Compose([
    A.Resize(height=IMAGE_SIZE, width=IMAGE_SIZE),
    A.Normalize(mean=NORM_MEAN, std=NORM_STD),
    ToTensorV2(),
])

print("Augmentation pipeline initialized.")


## 5. Streaming Datasets

Training streams from **training shards** with resampling and shard shuffling.

Validation samples are drawn from **training shards** with a per-shard cap (~20 samples
per shard) to ensure class diversity across all 134 classes. ScaleDF's dedicated `val`
split uses different source names with zero overlap to training classes, making it
unsuitable for fine-grained evaluation (only binary metrics would work on it).

Each sample's 134-class label is derived from the shard filename in its `__url__`.


In [ ]:
# ── Module 6: Streaming Datasets ──
import webdataset as wds

def extract_image(sample):
    """Extract a PIL Image from a webdataset sample."""
    for key in ("jpg", "jpeg", "png", "webp", "bmp", "tiff", "ppm"):
        if key in sample and isinstance(sample[key], Image.Image):
            return sample[key]
    for key, val in sample.items():
        if key.startswith("__"):
            continue
        if isinstance(val, Image.Image):
            return val
        if isinstance(val, bytes) and len(val) > 100:
            try:
                return Image.open(io.BytesIO(val))
            except Exception:
                continue
    return None

def _validate_image(img):
    """Convert to RGB numpy array, rejecting broken or tiny images."""
    if img is None:
        return None
    try:
        arr = np.array(img.convert("RGB"))
        if arr.ndim != 3 or arr.shape[2] != 3:
            return None
        if min(arr.shape[:2]) < 20:
            return None
        return arr
    except Exception:
        return None


class ScaleDFTrainDataset(torch.utils.data.IterableDataset):
    """Streaming training dataset with 134-class labels from shard filenames."""

    def __init__(self, urls, augmentation, url_to_label):
        super().__init__()
        self.urls = list(urls)
        self.augmentation = augmentation
        self.url_to_label = url_to_label

    def __iter__(self):
        # Worker-aware shard splitting to prevent data duplication
        urls = self.urls
        worker_info = torch.utils.data.get_worker_info()
        if worker_info is not None:
            n = worker_info.num_workers
            w = worker_info.id
            urls = self.urls[w::n] or self.urls  # fallback to all if split is empty

        pipe = (
            wds.WebDataset(
                urls,
                resampled=True,
                shardshuffle=True,
                handler=wds.warn_and_continue,
            )
            .shuffle(2000, handler=wds.warn_and_continue)
            .decode("pil", handler=wds.warn_and_continue)
        )

        for sample in pipe:
            # Derive 134-class label from the shard URL
            raw_url = sample.get("__url__", "")
            shard_filename = raw_url.split("?")[0].split("/")[-1]

            if shard_filename not in self.url_to_label:
                continue

            label = self.url_to_label[shard_filename]
            img = extract_image(sample)
            arr = _validate_image(img)
            if arr is None:
                continue

            try:
                pixel_values = self.augmentation(image=arr)["image"]
                yield {"pixel_values": pixel_values, "label": label}
            except Exception:
                continue


class ScaleDFValDataset(torch.utils.data.Dataset):
    """Fixed-size validation dataset sampled from training shards.

    Uses a per-shard cap to ensure class diversity across all 134 classes.
    Stores raw PIL images and applies val augmentations on __getitem__.
    """

    def __init__(self, urls, url_to_label, augmentation, max_samples=2000, max_per_shard=20):
        super().__init__()
        self.augmentation = augmentation
        self.items = []  # List of (PIL.Image, label_id)

        print(f"Collecting up to {max_samples} val samples (~{max_per_shard} per shard)...")

        for url in urls:
            if len(self.items) >= max_samples:
                break
            shard_count = 0
            shard_filename = url.split("?")[0].split("/")[-1]
            label = url_to_label.get(shard_filename)
            if label is None:
                continue

            pipe = (
                wds.WebDataset(url, shardshuffle=False, handler=wds.warn_and_continue)
                .decode("pil", handler=wds.warn_and_continue)
            )
            for sample in pipe:
                if shard_count >= max_per_shard or len(self.items) >= max_samples:
                    break
                img = extract_image(sample)
                if img is None:
                    continue
                try:
                    rgb = img.convert("RGB")
                    arr = np.array(rgb)
                    if arr.ndim != 3 or arr.shape[2] != 3 or min(arr.shape[:2]) < 20:
                        continue
                    self.items.append((rgb, label))
                    shard_count += 1
                except Exception:
                    continue

        # Report class distribution
        from collections import Counter
        dist = Counter(label for _, label in self.items)
        n_real = sum(v for k, v in dist.items() if k in real_class_ids)
        n_fake = sum(v for k, v in dist.items() if k in fake_class_ids)
        print(f"Collected {len(self.items)} val samples ({n_real} real, {n_fake} fake, {len(dist)} classes)")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        img, label = self.items[idx]
        arr = np.array(img)
        pixel_values = self.augmentation(image=arr)["image"]
        return {"pixel_values": pixel_values, "label": label}

print("Dataset classes defined.")


In [ ]:
# ── Instantiate Datasets ──
# Val is sampled from training shards (same 134 classes) so fine-grained metrics work.
# With 14M+ images in ScaleDF, 2000 val samples is a negligible overlap.
# Per-shard cap (~20) ensures class diversity across all 134 sources.

print("Creating training dataset (streaming)...")
train_ds = ScaleDFTrainDataset(train_urls, train_augmentations, url_to_label)

print("Creating validation dataset (sampling from training shards)...")
shuffled_urls = list(train_urls)
random.shuffle(shuffled_urls)
val_ds = ScaleDFValDataset(shuffled_urls, url_to_label, val_augmentations, max_samples=2000)

assert len(val_ds) > 0, (
    "Validation dataset is empty! Check your network connection and HF token. "
    "If rate-limited, try: huggingface-cli login"
)

print(f"\nDatasets ready! Val size: {len(val_ds)}")


## 6. Sanity Check

Verify the pipeline produces correctly shaped tensors with valid labels.


In [ ]:
# ── Module 7: Sanity Check ──
print("Pulling one training sample from the shard stream...")
sample = next(iter(train_ds))
print(f"  pixel_values shape: {sample['pixel_values'].shape}")
print(f"  pixel_values dtype: {sample['pixel_values'].dtype}")

label_id = sample["label"]
class_name = id2label[label_id]
is_fake = "Fake" if label_id in fake_class_ids else "Real"

print(f"  label ID:       {label_id}")
print(f"  class name:     {class_name}")
print(f"  binary group:   {is_fake}")

assert sample["pixel_values"].shape == (3, IMAGE_SIZE, IMAGE_SIZE), \
    f"Expected (3, {IMAGE_SIZE}, {IMAGE_SIZE}), got {sample['pixel_values'].shape}"

# Verify val dataset
val_sample = val_ds[0]
print(f"\n  val pixel_values shape: {val_sample['pixel_values'].shape}")
val_class = id2label[val_sample['label']]
print(f"  val class: {val_class}")

print("\nPipeline is working correctly!")


In [ ]:
# ── Visualize Training Samples ──
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 3, figsize=(12, 8))
stream = iter(train_ds)
for ax in axes.flat:
    s = next(stream)
    img = s["pixel_values"].permute(1, 2, 0).numpy()
    img = img * np.array(NORM_STD) + np.array(NORM_MEAN)  # denormalize
    label_name = id2label[s["label"]]
    binary = "Fake" if s["label"] in fake_class_ids else "Real"
    ax.imshow(np.clip(img, 0, 1))
    ax.set_title(f"{binary}: {label_name}", fontsize=9)
    ax.axis("off")
plt.suptitle("Augmented Training Samples (134-class)", fontsize=14)
plt.tight_layout()
plt.show()


## 7. Model & Training

**134-class EfficientNet-B4** fine-tuning with:
- **Cosine LR scheduler** with warmup
- **Label smoothing** for better calibration
- **Both fine-grained AND binary metrics** tracked during evaluation
- **Early stopping** on binary F1 (the ultimate production metric)

> The `collate_fn` is required to prevent HuggingFace Trainer from crashing
> when it tries to auto-remove columns that the model's `forward()` doesn't accept.


In [ ]:
# ── Module 8: Model & Training ──
from transformers import (
    AutoModelForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
import evaluate
from scipy.special import softmax

# ── Load model with 134-class head ──
print(f"Loading EfficientNet-B4 with {NUM_CLASSES} classes...")
model = AutoModelForImageClassification.from_pretrained(
    MODEL_ID,
    num_labels=NUM_CLASSES,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True,  # Replace default ImageNet head
)
print(f"Model loaded. Output head: {NUM_CLASSES} classes.")

# ── Metrics ──
clf_metrics = evaluate.combine(["accuracy", "f1", "precision", "recall"])

def compute_metrics(eval_pred):
    """Compute both fine-grained (134-class) and binary (Real/Fake) metrics."""
    logits, labels = eval_pred

    # 1. Fine-grained accuracy
    fine_preds = np.argmax(logits, axis=-1)
    fine_acc = np.mean(fine_preds == labels)

    # 2. Binary metrics — sum fake-class probabilities
    probs = softmax(logits, axis=-1)
    fake_probs = probs[:, fake_class_ids].sum(axis=-1)
    binary_preds = (fake_probs > 0.5).astype(int)
    binary_labels = np.isin(labels, fake_class_ids).astype(int)

    binary_results = clf_metrics.compute(
        predictions=binary_preds, references=binary_labels
    )

    return {
        "fine_grained_accuracy": fine_acc,
        "binary_accuracy": binary_results["accuracy"],
        "binary_f1": binary_results["f1"],
        "binary_precision": binary_results["precision"],
        "binary_recall": binary_results["recall"],
    }

# ── Collate function (prevents Trainer column-pruning crash) ──
def collate_fn(examples):
    pixel_values = torch.stack([ex["pixel_values"] for ex in examples])
    labels = torch.tensor([ex["label"] for ex in examples], dtype=torch.long)
    return {"pixel_values": pixel_values, "labels": labels}

# ── Training configuration ──
OUTPUT_DIR = "./deeptrace-efficientnet-v2"

# T4 supports fp16 only; A100+ supports bf16 (more stable)
use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    # Streaming: use max_steps (no epochs)
    max_steps=15000,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,

    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    save_total_limit=2,

    learning_rate=3e-5,
    weight_decay=1e-5,
    lr_scheduler_type="cosine",
    warmup_steps=750,
    label_smoothing_factor=0.1,

    metric_for_best_model="binary_f1",
    greater_is_better=True,
    load_best_model_at_end=True,

    fp16=use_fp16,
    bf16=use_bf16,

    # Set to True if you hit CUDA OOM on T4
    gradient_checkpointing=False,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    logging_steps=50,
    dataloader_num_workers=0,  # Prevents Kaggle multiprocessing deadlock
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print(f"Training for up to {training_args.max_steps:,} steps")
print(f"Eval every {training_args.eval_steps:,} steps")
print(f"Early stopping patience: 3 evaluations ({3 * training_args.eval_steps:,} steps)")
print(f"Mixed precision: {'bf16' if use_bf16 else 'fp16' if use_fp16 else 'none'}")


In [ ]:
# 🚀 Run Training
# To resume after a disconnect, change to: trainer.train(resume_from_checkpoint=True)
print("Starting EfficientNet-B4 fine-grained training on ScaleDF...")
trainer.train()


## 8. Evaluation

Both **fine-grained** (134-class) and **binary** (Real vs Fake) metrics, plus
confusion matrix and per-class breakdown.


In [ ]:
# ── Module 9: Evaluation ──
from sklearn.metrics import classification_report, confusion_matrix

print("Evaluating on validation set...")
metrics = trainer.evaluate()
for k, v in sorted(metrics.items()):
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

# Detailed predictions
predictions = trainer.predict(val_ds)
y_pred_fine = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

# Binary conversion
probs = softmax(predictions.predictions, axis=-1)
y_pred_binary = (probs[:, fake_class_ids].sum(axis=-1) > 0.5).astype(int)
y_true_binary = np.isin(y_true, fake_class_ids).astype(int)

print("\n" + "="*60)
print("BINARY CLASSIFICATION REPORT (Real vs Fake)")
print("="*60)
print(classification_report(
    y_true_binary, y_pred_binary,
    target_names=["Real", "Fake"], digits=4
))

cm = confusion_matrix(y_true_binary, y_pred_binary)
print("Binary Confusion Matrix (rows=true, cols=predicted):")
print(f"{'':>8}{'Real':>8}{'Fake':>8}")
for i, name in enumerate(["Real", "Fake"]):
    print(f"{name:>8}" + "".join(f"{v:>8}" for v in cm[i]))

fn = cm[1][0]  # Fake classified as Real
fp = cm[0][1]  # Real classified as Fake
print(f"\nFalse negatives (Fake → Real): {fn} / {cm[1].sum()}")
print(f"False positives (Real → Fake): {fp} / {cm[0].sum()}")

# Top-5 fine-grained accuracy
top5 = np.mean([
    y_true[i] in np.argsort(predictions.predictions[i])[-5:]
    for i in range(len(y_true))
])
print(f"\nFine-grained Top-1 accuracy: {np.mean(y_pred_fine == y_true):.4f}")
print(f"Fine-grained Top-5 accuracy: {top5:.4f}")


## 9. Save, Verify & Export

Saves the trained model + processor + class metadata. A round-trip reload
verifies the checkpoint is valid before exporting.

> The `class_metadata.json` file stores `real_class_ids` and `fake_class_ids` —
> the backend needs these to aggregate probabilities into Real/Fake.


In [ ]:
# ── Module 10: Save, Verify & Export ──
import json
import shutil

# 1. Save model + processor
trainer.save_model(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

# 2. Save class metadata for the backend
class_metadata = {
    "num_classes": NUM_CLASSES,
    "real_class_ids": real_class_ids,
    "fake_class_ids": fake_class_ids,
    "id2label": {str(k): v for k, v in id2label.items()},
    "label2id": label2id,
}
with open(os.path.join(OUTPUT_DIR, "class_metadata.json"), "w") as f:
    json.dump(class_metadata, f, indent=2)
print("Saved class_metadata.json")

# 3. Round-trip verification
print("\nVerifying saved checkpoint...")
_check_model = AutoModelForImageClassification.from_pretrained(OUTPUT_DIR)
_check_processor = AutoImageProcessor.from_pretrained(OUTPUT_DIR)

assert _check_model.config.num_labels == NUM_CLASSES, \
    f"Expected {NUM_CLASSES} labels, got {_check_model.config.num_labels}"
assert len(_check_model.config.id2label) == NUM_CLASSES, \
    f"Expected {NUM_CLASSES} id2label entries, got {len(_check_model.config.id2label)}"
assert list(_check_processor.image_mean) == list(NORM_MEAN), \
    f"Norm mean mismatch: saved={_check_processor.image_mean}, expected={NORM_MEAN}"
assert list(_check_processor.image_std) == list(NORM_STD), \
    f"Norm std mismatch: saved={_check_processor.image_std}, expected={NORM_STD}"

print(f"  num_labels:  {_check_model.config.num_labels}")
print(f"  norm mean:   {_check_processor.image_mean}")
print(f"  norm std:    {_check_processor.image_std}")
print("  ✓ Checkpoint verified!")
del _check_model, _check_processor

# 4. Zip and download
zip_filename = "deeptrace_efficientnet_v2"
shutil.make_archive(zip_filename, "zip", OUTPUT_DIR)
print(f"\nCreated {zip_filename}.zip")

try:
    from google.colab import files
    files.download(f"{zip_filename}.zip")
    print("Downloading via browser...")
except Exception:
    print("Running on Kaggle: Download the zip from the 'Output' tab.")

print("\nExtract into backend/models/deeptrace-efficientnet-v2 and update DEEPTRACE_MODEL in .env!")
